# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA (Low Rank Adaptation)
* Model: DistilBERT base uncased for sequence classification
* Evaluation approach: Hugging Face Trainer with accuracy metric on the validation set
* Fine-tuning dataset: AG News dataset from the Hugging Face datasets library

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [4]:
# Load the ag_news dataset
# See: https://huggingface.co/datasets/szhuggingface/ag_news

from datasets import load_dataset

# The ag_news dataset only has a train split, so we use the train_test_split method to split it into train and test
repo_name = "szhuggingface/ag_news" 
dataset = load_dataset(repo_name, split="train_1_12k").train_test_split(
    test_size=0.2, shuffle=True, seed=23
)

splits = ["train", "test"]

# View the dataset characteristics
dataset["train"]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Generating val_full split:   0%|          | 0/24000 [00:00<?, ? examples/s]

Generating val1 split:   0%|          | 0/12000 [00:00<?, ? examples/s]

Generating val2 split:   0%|          | 0/12000 [00:00<?, ? examples/s]

Generating train_full split:   0%|          | 0/96000 [00:00<?, ? examples/s]

...

Generating train_1_12k split:   0%|          | 0/12000 [00:00<?, ? examples/s]

Generating train_1_6k split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating train_1_3k split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating train_1_1.5k split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 9600
})

In [5]:
# Inspect the first example.
dataset["train"][0]

{'text': 'Sane people won #39;t miss it Yes, Greece, you did good. Defied the doomsayers. Put on a good show. Blabbedy blah blah blah. But after three weeks here, most of them spent on the shuttle bus from the volleyball to taekwondo ',
 'label': 1}

## Pre-process datasets

Now we are going to process our datasets by converting all the text into tokens for our models.

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Let's use a lambda function to tokenize all the examples
tokenized_dataset = {}
for split in splits:
    tokenized_dataset[split] = dataset[split].map(
        lambda x: tokenizer(x["text"], truncation=True), batched=True
    )

# Inspect the available columns in the dataset
tokenized_dataset["train"]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 9600
})

## Load and set up the model

In this case we freeze the DistilBERT base model and train only the classification head for the downstream task.

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4,
    id2label={0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"},  # For converting predictions to strings
    label2id={"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3},
)

# Freeze all the parameters of the base model
# Hint: Check the documentation at https://huggingface.co/transformers/v4.2.2/training.html
for param in model.base_model.parameters():
    param.requires_grad = False

model.classifier

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Linear(in_features=768, out_features=4, bias=True)

## Let's train it!

Now it's time to train our model. We'll use the `Trainer` class.

First we'll define a function to compute our accuracy metreic then we make the `Trainer`.

In this instance, we will fill in some of the training arguments

In [8]:
import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}


# The HuggingFace Trainer class handles the training and eval loop for PyTorch for us.
# Read more about it here https://huggingface.co/docs/transformers/main_classes/trainer
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./data/ag_news_text_classifier",
        # Set the learning rate
        learning_rate=2e-3,
        # Set the per device train batch size and eval batch size
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        # Evaluate and save the model after each epoch
        evaluation_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.405800,0.366583,0.882917
2,0.329900,0.316044,0.897083
3,0.273800,0.270710,0.904167


Checkpoint destination directory ./data/ag_news_text_classifier/checkpoint-600 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./data/ag_news_text_classifier/checkpoint-1200 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./data/ag_news_text_classifier/checkpoint-1800 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=1800, training_loss=0.32487272474500867, metrics={'train_runtime': 108.7669, 'train_samples_per_second': 264.786, 'train_steps_per_second': 16.549, 'total_flos': 729681294166272.0, 'train_loss': 0.32487272474500867, 'epoch': 3.0})

In [9]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

## Evaluate the model

Evaluating the model is as simple as calling the evaluate method on the trainer object. This will run the model on the test set and compute the metrics we specified in the compute_metrics function.

In [10]:
# Show the performance of the model on the test set
trainer.evaluate()

{'eval_loss': 0.2707095742225647,
 'eval_accuracy': 0.9041666666666667,
 'eval_runtime': 6.5782,
 'eval_samples_per_second': 364.843,
 'eval_steps_per_second': 22.803,
 'epoch': 3.0}

In [11]:
trainer.model = model
original_performance = trainer.evaluate()
print("Original Fine Tuned Model:", original_performance)

Original Fine Tuned Model: {'eval_loss': 0.2707095742225647, 'eval_accuracy': 0.9041666666666667, 'eval_runtime': 6.7138, 'eval_samples_per_second': 357.472, 'eval_steps_per_second': 22.342, 'epoch': 3.0}


### View the results

Let's look at a few examples

In [12]:
# Make a dataframe with the predictions and the text and the labels
import pandas as pd

items_for_manual_review = tokenized_dataset["test"].select(
    [0, 1, 22, 31, 43, 292, 448, 487]
)

results = trainer.predict(items_for_manual_review)
df = pd.DataFrame(
    {
        "text": [item["text"] for item in items_for_manual_review],
        "predictions": results.predictions.argmax(axis=1),
        "labels": results.label_ids,
    }
)
# Show all the cell
pd.set_option("display.max_colwidth", None)
df

,text,predictions,labels
0,Ben Wallace Recovering After Appendectomy (AP) AP - Detroit Pistons center Ben Wallace had an emergency appendectomy and was recovering Thursday.,1,1
1,"OPEC: Group Doing All It Can To Stabilize Oil Prices OPEC President Purnomo Yusgiantoro highlighted the impact of the current high oil prices, saying the direct contribution of the increase in global oil prices to the economic slowdown",2,2
2,"Lewinsky and Loos dish on kiss-and-tell after Clinton and Beckham (AFP) AFP - Kiss-and-tell queens Monica Lewinsky and Rebecca Loos, in a debate on media payments for interviews, came clean about dishing the dirt on Bill Clinton and David Beckham, but the former White House intern claimed her sex story sat on higher moral ground.",0,0
3,"SuperSonics Top Lakers 108-93 (AP) AP - Rashard Lewis scored a season-high 37 points, and the Seattle SuperSonics improved to 18-4 with a 108-93 win over the Los Angeles Lakers Tuesday night.",1,1
4,"eBay bidding ends at \$28,000 for 10-year-old sandwich with image HOLLYWOOD, Fla. Half of a ten-year-old grilled cheese sandwich said to bear the image of the Virgin Mary has sold on eBay for 28-thousand dollars.",2,3
5,Western Leaders Criticize Putin More than 100 American and European foreign policy experts signed a letter protesting against Russian President Vladimir Putin #39;s leadership.,0,0
6,Spin doctors must deal with truth So the 600-page Olympic and Paralympics candidate file for London 2012 is now with the International Olympic Committee and Sebastian Coe has told us we must Make Britain Proud and Back the Bid,1,1
7,Todd MacCulloch Retires Philadelphia 76ers President and General Manager Billy King announced today that center Todd MacCulloch will retire from the game of basketball.,1,1


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

### Creating a PEFT Config
#### Initiate a LoRA config
The PEFT config specifies the adapter configuration for the parameter-efficient fine-tuning process. The base class for this is a PeftConfig. We will use a LoraConfig, the subclass used for low rank adaptation (LoRA).






In [13]:
import torch
from peft import LoraConfig, LoraModel, TaskType

device = torch.accelerator.current_accelerator().type if hasattr(torch, "accelerator") else "cuda"
config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.01,
    # target_modules=["q_lin", "v_lin", ...]  # optionally indicate target modules
)

### Converting a Transformers Model into a PEFT Model
Once we have a PEFT config object, we can load a Hugging Face transformers model as a PEFT model by first loading the pre-trained model as usual:

In [14]:
#Converting a Transformers Model into a PEFT Model
from peft import get_peft_model
#Then using get_peft_model() to get a trainable PEFT model (using the LoRA config instantiated previously):
lora_model = get_peft_model(model, config)

### Training with a PEFT Model
After calling get_peft_model(), we can then use the resulting lora_model in a training process (Hugging Face Trainer).



In [15]:
#Training with a PEFT Model
trainer = Trainer(
    model=lora_model,
    args=TrainingArguments(
        output_dir="./data/ag_news_PEFT_text_classifier",
        # Set the learning rate
        learning_rate=2e-3,
        # Set the per device train batch size and eval batch size
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        # Evaluate and save the model after each epoch
        evaluation_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
    ),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.376100,0.300121,0.905833
2,0.290300,0.295954,0.911667
3,0.192600,0.269537,0.922083


TrainOutput(global_step=1800, training_loss=0.264512136247423, metrics={'train_runtime': 208.2804, 'train_samples_per_second': 138.275, 'train_steps_per_second': 8.642, 'total_flos': 742221966021120.0, 'train_loss': 0.264512136247423, 'epoch': 3.0})

### Checking Trainable Parameters of a PEFT Model
A helpful way to check the number of trainable parameters with the current config is the print_trainable_parameters() method:



In [16]:
#Checking Trainable Parameters of a PEFT Model
lora_model.print_trainable_parameters()

trainable params: 1,334,792 || all params: 67,697,672 || trainable%: 1.9716955702701269


In [17]:
trainer.model = lora_model
fine_tuned_performance = trainer.evaluate()
print("PEFT Model:", fine_tuned_performance)

PEFT Model: {'eval_loss': 0.2695373594760895, 'eval_accuracy': 0.9220833333333334, 'eval_runtime': 7.5154, 'eval_samples_per_second': 319.346, 'eval_steps_per_second': 19.959, 'epoch': 3.0}


### View the results

Let's look at a few examples

In [18]:
# Make a dataframe with the predictions and the text and the labels
import pandas as pd

items_for_manual_review = tokenized_dataset["test"].select(
    [0, 1, 22, 31, 43, 292, 448, 487]
)

results = trainer.predict(items_for_manual_review)
df = pd.DataFrame(
    {
        "text": [item["text"] for item in items_for_manual_review],
        "predictions": results.predictions.argmax(axis=1),
        "labels": results.label_ids,
    }
)
# Show all the cell
pd.set_option("display.max_colwidth", None)
df

,text,predictions,labels
0,Ben Wallace Recovering After Appendectomy (AP) AP - Detroit Pistons center Ben Wallace had an emergency appendectomy and was recovering Thursday.,1,1
1,"OPEC: Group Doing All It Can To Stabilize Oil Prices OPEC President Purnomo Yusgiantoro highlighted the impact of the current high oil prices, saying the direct contribution of the increase in global oil prices to the economic slowdown",2,2
2,"Lewinsky and Loos dish on kiss-and-tell after Clinton and Beckham (AFP) AFP - Kiss-and-tell queens Monica Lewinsky and Rebecca Loos, in a debate on media payments for interviews, came clean about dishing the dirt on Bill Clinton and David Beckham, but the former White House intern claimed her sex story sat on higher moral ground.",0,0
3,"SuperSonics Top Lakers 108-93 (AP) AP - Rashard Lewis scored a season-high 37 points, and the Seattle SuperSonics improved to 18-4 with a 108-93 win over the Los Angeles Lakers Tuesday night.",1,1
4,"eBay bidding ends at \$28,000 for 10-year-old sandwich with image HOLLYWOOD, Fla. Half of a ten-year-old grilled cheese sandwich said to bear the image of the Virgin Mary has sold on eBay for 28-thousand dollars.",3,3
5,Western Leaders Criticize Putin More than 100 American and European foreign policy experts signed a letter protesting against Russian President Vladimir Putin #39;s leadership.,0,0
6,Spin doctors must deal with truth So the 600-page Olympic and Paralympics candidate file for London 2012 is now with the International Olympic Committee and Sebastian Coe has told us we must Make Britain Proud and Back the Bid,1,1
7,Todd MacCulloch Retires Philadelphia 76ers President and General Manager Billy King announced today that center Todd MacCulloch will retire from the game of basketball.,1,1


###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

### Saving a Trained PEFT Model
Once a PEFT model has been trained, the standard Hugging Face save_pretrained() method can be used to save the weights locally. 



In [19]:
# Saving the model
lora_model.save_pretrained("/tmp/my-model")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

### Loading a Saved PEFT Model
Because we have only saved the adapter weights and not the full model weights, we can't use from_pretrained() with the regular Transformers class. Instead, we need to use the PEFTModel.



In [20]:
#Loading a Saved PEFT Model
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

# Reloading the model
reloaded_model = PeftModel.from_pretrained(base_model, "/tmp/my-model")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Evaluate the reloaded model

In [21]:
peft_trainer = Trainer(
    model=reloaded_model,
    args=TrainingArguments(output_dir="./tmp_eval"),
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

loaded_performance = peft_trainer.evaluate()

print("Reloaded PEFT Model:", loaded_performance)

Reloaded PEFT Model: {'eval_loss': 0.2695373594760895, 'eval_accuracy': 0.9220833333333334, 'eval_runtime': 6.8631, 'eval_samples_per_second': 349.695, 'eval_steps_per_second': 43.712}


### Compare the Model Performance
Compare the Fine-tuned Model Performance to the Original Model with Accuracy Metric on the Validation Set

In [22]:
print("Original Model Accuracy:", original_performance["eval_accuracy"])
print("PEFT Model Accuracy:", fine_tuned_performance["eval_accuracy"])
print("Reloaded PEFT Accuracy:", loaded_performance["eval_accuracy"])

Original Model Accuracy: 0.9041666666666667
PEFT Model Accuracy: 0.9220833333333334
Reloaded PEFT Accuracy: 0.9220833333333334


Conclusion: LoRA based parameter efficient fine tuning improved the model accuracy from 0.905 to 0.922 while training only about two percent of the model parameters.